# Databricks

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/databricks.ipynb)

Databricks is a managed lakehouse platform built on Apache Spark, Delta Lake, and Unity Catalog.
Platform-only features (Unity Catalog grants, Jobs, Lakeflow pipelines, system tables) appear as
reference snippets; the mechanics underneath - Delta commits and time travel, `MERGE`, change
data feed, incremental ingestion, expectations, task DAGs, and cost attribution - run locally
with `deltalake`, PySpark, and plain Python.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1", "deltalake": "deltalake>=1.0", "pyarrow": "pyarrow>=15"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile

from pyspark.sql import SparkSession, functions as F

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this interpreter
WORK_DIR = tempfile.mkdtemp(prefix="databricks-lessons-")

spark = (
    SparkSession.builder
    .appName("databricks-lessons")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("local Spark", spark.version, "standing in for a Databricks cluster")

## What is Databricks?

Databricks packages Spark with Delta Lake, a governance layer (Unity Catalog), SQL warehouses,
workflow orchestration, and ML tooling. The core promise is the **lakehouse**: ACID tables, schema
enforcement, and time travel on cheap object storage, with one copy of data serving BI, data
science, and streaming.

```text
cloud storage (S3/ADLS/GCS)
        |
   Delta / Iceberg tables          <- open formats, one copy of data
        |
Spark + Photon  |  SQL warehouses  |  MLflow / serving
        |
   Unity Catalog: catalogs.schemas.tables + lineage + grants
```

Think in three planes:

+ **Control plane** (Databricks-managed): workspace UI, notebooks, job scheduler, Unity Catalog
  metadata, cluster manager.
+ **Compute plane**: VMs in your cloud account (classic) or in a Databricks account (serverless)
  where code executes.
+ **Storage plane**: object storage in your account holding the Parquet files and `_delta_log`.

A nightly job: the scheduler (control plane) requests a job cluster; driver and executors start
as VMs in your account (compute), read Delta files from your bucket, and write new Parquet files
plus a log commit (storage). You pay **DBUs** to Databricks for software time and, on classic
compute, your cloud provider separately for VMs and disks; serverless folds both into the DBU
rate.

Common mistakes: assuming Databricks holds your data (it sees metadata and query text; the data
stays in your account), and treating the DBU price as the whole bill - an idle classic cluster
keeps the VM bill running. Ask which plane a cost or failure belongs to before proposing a fix.

## How does the Databricks architecture work?

+ **Control plane** - the Databricks-managed layer: workspace UI, notebooks, job scheduler, Unity Catalog metadata, cluster manager. Databricks operates it.
+ **Data plane (compute plane)** - clusters and SQL warehouses that run in your cloud account/network, next to your storage. Data does not have to leave your account.
+ **Serverless compute** - Databricks-managed compute in the Databricks account; no cluster to size, faster startup, per-second billing.
+ **Storage** - object storage in your account (S3, ADLS Gen2, GCS), accessed through Unity Catalog external locations and storage credentials rather than hard-coded keys.
+ **Notebooks and jobs** - interactive work and scheduled production work use the same Spark engine and the same tables.

```text
Workspace (control plane)
   |  REST API / CLI / Terraform / bundles
   v
Compute plane (your cloud account)
   +-- all-purpose cluster (interactive, shared)
   +-- job cluster (ephemeral, per-run)
   +-- SQL warehouse (BI and SQL)
   +-- serverless (Databricks account)
   |
   v
Object storage: s3://acme-lake/bronze|silver|gold
```

A workspace attaches to one Unity Catalog **metastore** per region, shared by every workspace
in that region - so a table created in one workspace is visible in another. Point at any
component and name its plane: the notebook and its history (control), the driver, executors, and
shuffle files (compute), the `_delta_log` commits and Parquet files (storage). An unreachable
bucket is compute failing to reach storage; resizing the cluster cannot fix it.

Common mistake: putting cloud keys in a notebook. Use a cloud identity behind a **storage
credential**, wrapped by an **external location**, with Unity Catalog deciding who may use it.

## What cluster types exist and when do you use each?

| Cluster | Lifetime | Used for | Cost profile |
|---|---|---|---|
| All-purpose | long-lived, shared | interactive notebooks, exploration, ad-hoc | expensive per DBU; idle cost if not auto-stopped |
| Job cluster | created per job run | production pipelines | cheapest for scheduled work, isolated per run |
| SQL warehouse | managed, auto-stop | BI, dashboards, SQL users | sized by warehouse size and clusters |
| Serverless (jobs/SQL/notebooks) | managed | bursty or latency-sensitive workloads | no cluster management, per-second billing |

+ Run production pipelines on **job clusters** (or serverless jobs), never on a developer's all-purpose cluster.
+ Use **all-purpose** clusters for exploration and pair them with aggressive auto-stop.
+ Use **SQL warehouses** for BI so warehouse sizing and concurrency are independent of Spark jobs.
+ Give each workload its own cluster **access mode** and policy when governance requires isolation.

```json
{
  "cluster_name": "etl-daily",
  "spark_version": "auto:latest-lts",
  "node_type_id": "m5d.4xlarge",
  "autoscale": {"min_workers": 4, "max_workers": 32},
  "aws_attributes": {"availability": "SPOT_WITH_FALLBACK", "zone_id": "auto"},
  "autotermination_minutes": 20,
  "custom_tags": {"team": "data-platform", "cost-center": "analytics"},
  "spark_conf": {"spark.sql.adaptive.enabled": "true"}
}
```

The engine is the same; lifetime and billing differ. An **all-purpose** cluster bills the
premium rate for every minute it is up, including idle minutes. A **job cluster** is created per
run at the cheaper jobs rate and never idles. A **SQL warehouse** is sized for SQL concurrency.
**Serverless** removes cluster management and bills per second with startup in seconds.

Common mistakes: production work on a shared all-purpose cluster (a runaway notebook slows the
pipeline and a forgotten session bills all weekend), and using a warehouse for Spark work or a
general cluster for dashboards. Moving a workload to the right type is often the largest saving
available.

## How do you size and configure a cluster?

+ **Workers** - memory-optimized (r-family) for shuffles and caching; compute-optimized (c-family) for CPU-heavy transformation; storage-optimized (i/d-family) when spilling or caching to local disk matters.
+ **Autoscaling** - scale within a range; it reacts to pending tasks, so it is not a substitute for fixing skew.
+ **Spot instances with fallback** - large savings for batch; avoid for latency-critical or stateful streaming drivers.
+ **Driver size** - the driver collects results and runs the DAG; `collect()` on a large DataFrame still kills the driver.
+ **Photon** - enable it for SQL/DataFrame heavy workloads; it does not accelerate row-at-a-time Python UDFs.
+ **Auto-stop** - set 10-20 minutes on interactive clusters; the biggest source of accidental spend is an idle cluster.

A cluster is one driver plus N workers, each running one executor, so worker count and size
set total parallelism. Autoscaling adjusts workers within a range based on pending tasks (job
compute scales down after roughly 40 seconds of underuse, all-purpose after roughly 150); do not
also enable Spark dynamic allocation, because the two managers fight. The driver is always
on-demand, even on spot clusters.

Common mistakes: an undersized driver for a notebook that `collect()`s large results, and
expecting autoscaling to fix skew - more workers cannot split one hot partition.

## What is Unity Catalog and how does the three-level namespace work?

```text
catalog.schema.table
acme.silver.orders
acme.gold.fct_orders
acme.bronze.orders_raw
```

+ **Metastore** - one per region (typically); attaches to workspaces.
+ **Catalog** - the top-level container, often per environment (`dev`, `prod`) or per domain.
+ **Schema** - database-level grouping of tables, views, volumes, functions, models.
+ **Volumes** - governed storage for files that are not tables (`/Volumes/acme/landing/orders/`).
+ **External locations and storage credentials** - central place to grant access to cloud storage; notebooks no longer need access keys.

```sql
CREATE CATALOG IF NOT EXISTS acme;
CREATE SCHEMA IF NOT EXISTS acme.silver;

CREATE TABLE IF NOT EXISTS acme.silver.orders (
    order_id      STRING NOT NULL,
    customer_id   STRING,
    order_ts      TIMESTAMP,
    amount        DECIMAL(18, 2),
    status        STRING
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
```

Unity Catalog replaces Hive's `database.table` with `catalog.schema.table`. Because the catalog
is part of the name, one cluster can read `dev.silver.orders` and `prod.silver.orders` with
completely independent grants.

File access follows a chain: a cloud identity is registered as a **storage credential**, an
**external location** names a path and points at that credential, and only then can tables or
volumes be created there. Grants mirror the chain (`CREATE EXTERNAL TABLE` on the location,
`READ FILES` / `WRITE FILES` for direct path access); managed tables need no per-path grant.

Common mistakes: hard-coded paths with embedded keys (bypassing governance) and two-level names
carried over from Hive - production jobs should always use fully qualified names.

## How do you govern data with Unity Catalog?

```sql
-- role-style grants at catalog/schema/table level
GRANT USE CATALOG ON CATALOG acme TO `analytics`;
GRANT USE SCHEMA ON SCHEMA acme.gold TO `analytics`;
GRANT SELECT ON TABLE acme.gold.fct_orders TO `analytics`;

-- column mask: analysts see a hashed email
CREATE FUNCTION acme.security.mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('pii-readers') THEN email
            ELSE sha2(email, 256) END;

ALTER TABLE acme.gold.dim_customer
ALTER COLUMN email SET MASK acme.security.mask_email;

-- row filter: each regional team sees only its region
CREATE FUNCTION acme.security.region_filter(region STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('global-readers')
       OR is_account_group_member(CONCAT('region-', region));

ALTER TABLE acme.gold.fct_orders
SET ROW FILTER acme.security.region_filter ON (region);
```

Also know **attribute-based access** with tags, **lineage** (table -> notebook/job -> dashboard), **audit logs** in `system.access.audit`, and **data contracts** via constraints plus enforced schemas.

A **row filter** is a SQL UDF returning a boolean, bound with `SET ROW FILTER`; a **column
mask** returns a value of the column's type, bound with `SET MASK`. Both run in the engine, so a
user with `SELECT` still sees only filtered rows and masked values; `is_account_group_member()`
branches on the caller. They restrict some optimizations such as pushdown, so keep the functions
simple (a `CASE`, not a lookup subquery).

Grants are hierarchical: `USE CATALOG` and `USE SCHEMA` are needed before `SELECT` on a table
does anything - the usual reason a grant "does not work". `MANAGE` delegates granting.
Attribute-based policies on a catalog or schema scale better than per-table filters and masks.

## How does Delta Lake work on Databricks?

Delta is Parquet files plus a transaction log (`_delta_log`). Every write appends a JSON commit
listing files added and removed; readers replay the log (with a Parquet checkpoint about every
ten commits) to get the current snapshot. A commit either fully lands or does not, which gives
ACID on object storage; concurrent writers use optimistic concurrency and retry on conflict.

`MERGE` is the upsert primitive. It is idempotent when the source has one row per key and the
update is conditional on recency, so a rerun changes nothing. The cell below uses `deltalake`
(delta-rs), which reads and writes the same table format Databricks does:

In [ ]:
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

orders_path = os.path.join(WORK_DIR, "silver_orders")
write_deltalake(
    orders_path,
    pa.table({"order_id": ["o1", "o2", "o3"], "status": ["placed"] * 3, "amount": [120.0, 84.5, 30.0], "source_seq": [1, 1, 1]}),
    configuration={"delta.enableChangeDataFeed": "true"},
)

staged = pa.table({  # o2 moved forward, o3's change is older than what we have, o4 is new
    "order_id": ["o2", "o3", "o4"],
    "status": ["paid", "cancelled", "placed"],
    "amount": [84.5, 30.0, 260.0],
    "source_seq": [2, 0, 1],
})


def merge_orders(source):
    return (
        DeltaTable(orders_path)
        .merge(source=source, predicate="t.order_id = s.order_id", source_alias="s", target_alias="t")
        .when_matched_update_all(predicate="s.source_seq > t.source_seq")
        .when_not_matched_insert_all()
        .execute()
    )


first = merge_orders(staged)
snapshot = DeltaTable(orders_path).to_pyarrow_table().sort_by("order_id")
second = merge_orders(staged)  # the same batch again

print({k: first[k] for k in ("num_target_rows_inserted", "num_target_rows_updated")})
print(snapshot.select(["order_id", "status", "source_seq"]).to_pylist())
assert dict(zip(snapshot["order_id"].to_pylist(), snapshot["status"].to_pylist())) == {
    "o1": "placed", "o2": "paid", "o3": "placed", "o4": "placed"}
assert second["num_target_rows_inserted"] == 0 and second["num_target_rows_updated"] == 0
assert DeltaTable(orders_path).to_pyarrow_table().sort_by("order_id").equals(snapshot)

Time travel reads the log at an older version, and `DESCRIBE HISTORY` lists the commits. The
window is bounded by log retention (about 30 days) and by `VACUUM`, which deletes unreferenced
data files older than `delta.deletedFileRetentionDuration` (7 days by default):

```sql
-- time travel and restore
SELECT * FROM acme.silver.orders VERSION AS OF 42;
SELECT * FROM acme.silver.orders TIMESTAMP AS OF '2026-10-01T00:00:00Z';
RESTORE TABLE acme.silver.orders TO VERSION AS OF 42;

-- inspect history and table details
DESCRIBE HISTORY acme.silver.orders;
DESCRIBE DETAIL acme.silver.orders;
```

In [ ]:
table = DeltaTable(orders_path)
for commit in sorted(table.history(), key=lambda h: h["version"]):
    print(commit["version"], commit["operation"])

version_zero = DeltaTable(orders_path, version=0).to_pyarrow_table()
assert version_zero.num_rows == 3 and set(version_zero["status"].to_pylist()) == {"placed"}

# Files that only old versions reference: VACUUM with zero retention would delete them,
# and time travel to those versions would then fail. Dry run only.
stale_files = table.vacuum(retention_hours=0, enforce_retention_duration=False, dry_run=True)
print("files a zero-retention VACUUM would delete:", len(stale_files))
assert len(stale_files) >= 1
assert table.vacuum(retention_hours=168, dry_run=True) == []  # nothing is a week old yet

With deletion vectors, `MERGE`, `UPDATE`, and `DELETE` mark rows deleted instead of rewriting
files; `REORG TABLE ... APPLY (PURGE)` applies them physically before a later `VACUUM`. A merge
source with duplicate keys fails on Databricks because the outcome would be ambiguous -
deduplicate in staging first.

## How do you run incremental ingestion with Auto Loader?

Auto Loader (`format("cloudFiles")`) discovers new files by directory listing or cloud
notifications, records processed files in a RocksDB-backed state store under the checkpoint, and
keeps the inferred schema in a separate schema location:

```python
raw = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/acme/bronze/checkpoints/orders_schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", 1000)
    .load("s3://acme-raw/orders/")
)
(
    raw.writeStream
    .option("checkpointLocation", "/Volumes/acme/bronze/checkpoints/orders")
    .trigger(availableNow=True)          # catch up on everything new, then stop
    .toTable("acme.bronze.orders_raw")
)
```

Schema evolution modes: `addNewColumns` (default) records the new column and fails the batch so
a restart picks it up - run under a job with retries; `addNewColumnsWithTypeWidening` also widens
int to long; `rescue` never evolves and puts unexpected fields in `_rescued_data`;
`failOnNewColumns` waits for a human; `none` ignores them.

Open-source Spark's file source has the same checkpointed, exactly-once progress. The cell lands
two files, ingests them with `availableNow`, lands a third, and ingests again - only the new file
is read:

In [ ]:
import json

landing = os.path.join(WORK_DIR, "landing", "orders")
bronze_path = os.path.join(WORK_DIR, "bronze", "orders_raw")
checkpoint = os.path.join(WORK_DIR, "checkpoints", "bronze_orders")
os.makedirs(landing)


def land(name, rows):
    with open(os.path.join(landing, name), "w") as handle:
        handle.write("\n".join(json.dumps(row) for row in rows))


def ingest():
    query = (
        spark.readStream
        .schema("order_id string, customer_id string, status string, amount double, order_ts timestamp")
        .json(landing)
        .withColumn("ingested_at", F.current_timestamp())
        .withColumn("source_file", F.input_file_name())
        .writeStream.format("parquet")
        .option("checkpointLocation", checkpoint)
        .trigger(availableNow=True)
        .start(bronze_path)
    )
    query.awaitTermination()
    return spark.read.parquet(bronze_path).count()


land("2026-10-04-0900.json", [
    {"order_id": "o1", "customer_id": "c1", "status": "placed", "amount": 120.0, "order_ts": "2026-10-04T09:00:00"},
    {"order_id": "o2", "customer_id": "c2", "status": "placed", "amount": 84.5, "order_ts": "2026-10-04T09:05:00"},
])
land("2026-10-04-0915.json", [
    {"order_id": "o2", "customer_id": "c2", "status": "paid", "amount": 84.5, "order_ts": "2026-10-04T09:12:00"},
    {"order_id": "o3", "customer_id": "c1", "status": "placed", "amount": -5.0, "order_ts": "2026-10-04T09:14:00"},
])
print("bronze rows after run 1:", ingest())

land("2026-10-04-0930.json", [
    {"order_id": "o4", "customer_id": None, "status": "placed", "amount": 42.0, "order_ts": "2026-10-04T09:31:00"},
])
bronze_rows = ingest()
print("bronze rows after run 2:", bronze_rows)
assert bronze_rows == 5  # each file was read exactly once

Common mistakes: two streams sharing one checkpoint, an oversized `maxFilesPerTrigger` that turns
the first batch into a multi-hour backfill, and treating `_rescued_data` as handled when nobody
alerts on it.

## How do you build a medallion pipeline (bronze, silver, gold)?

```text
bronze  raw append-only, Auto Loader, no business logic, keeps history
  |
silver  cleaned, deduplicated, typed, conformed keys; MERGE upserts
  |
gold    facts, dimensions, aggregates that BI and ML read
```

Bronze is raw and append-only (plus ingestion metadata), silver is deduplicated, typed, and
validated at business grain, gold is shaped for consumption. The layering buys replayability: fix
a silver bug and rebuild silver from bronze, because bronze still holds everything. Continuing
from the bronze table above:

In [ ]:
from pyspark.sql import Window

bronze = spark.read.parquet(bronze_path)

latest_first = Window.partitionBy("order_id").orderBy(F.col("order_ts").desc(), F.col("ingested_at").desc())
silver_orders = (
    bronze
    .withColumn("rn", F.row_number().over(latest_first))
    .where("rn = 1")
    .drop("rn", "source_file")
    .withColumn("order_date", F.to_date("order_ts"))
)
gold_daily = silver_orders.groupBy("order_date", "status").agg(
    F.count("*").alias("orders"), F.sum("amount").alias("revenue")
)

silver_orders.select("order_id", "status", "amount").orderBy("order_id").show()
gold_daily.show()
assert silver_orders.count() == 4  # o2's two versions collapsed to the latest
assert silver_orders.where("order_id = 'o2'").first()["status"] == "paid"

Common mistakes: business logic in bronze (the raw record is lost), a gold table that is a full
copy of silver (dashboards pay silver's scan cost), and bronze with no retention plan.

## What are Lakeflow Declarative Pipelines (Delta Live Tables)?

You declare datasets and quality rules; the runtime derives the dependency graph, orders and
retries updates, and records metrics in an event log. A **streaming table** processes new data
incrementally; a **materialized view** recomputes an aggregate over a changing source. Newer
runtimes expose `from pyspark import pipelines as dp` alongside `import dlt`:

```python
import dlt
from pyspark.sql import functions as F

@dlt.table(name="silver_orders")
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_fail("valid_amount", "amount >= 0")
def silver_orders():
    return (
        dlt.read_stream("bronze_orders")
        .withColumn("order_date", F.to_date("order_ts"))
        .dropDuplicates(["order_id"])
    )
```

`expect` records a metric and keeps the row, `expect_or_drop` drops and counts it,
`expect_or_fail` stops the update. The same semantics in plain PySpark, run against the silver
frame above:

In [ ]:
def apply_expectations(df, expect=None, expect_or_drop=None, expect_or_fail=None):
    """Mimic pipeline expectations: return the surviving rows and per-rule failure counts."""
    metrics = {}
    for name, rule in (expect or {}).items():
        metrics[name] = df.where(f"NOT coalesce({rule}, false)").count()
    for name, rule in (expect_or_fail or {}).items():
        failures = df.where(f"NOT coalesce({rule}, false)").count()
        if failures:
            raise ValueError(f"expectation {name!r} failed for {failures} row(s)")
    for name, rule in (expect_or_drop or {}).items():
        metrics[name] = df.where(f"NOT coalesce({rule}, false)").count()
        df = df.where(f"coalesce({rule}, false)")
    return df, metrics


checked, metrics = apply_expectations(
    silver_orders,
    expect={"has_customer": "customer_id IS NOT NULL"},
    expect_or_drop={"valid_amount": "amount >= 0"},
)
print("metrics:", metrics, "| rows kept:", checked.count())
assert metrics == {"has_customer": 1, "valid_amount": 1} and checked.count() == 3

try:
    apply_expectations(silver_orders, expect_or_fail={"valid_amount": "amount >= 0"})
except ValueError as error:
    print("update stopped:", error)

Common mistakes: `expect_or_drop` with no quarantine table (failures live only in a metric),
a materialized view where a streaming table belongs (full recompute each update), and mixing
`read` and `read_stream` so an incremental pipeline recomputes everything.

## How do you orchestrate jobs and workflows?

A job is a DAG of tasks (notebook, Python wheel, SQL, pipeline, ...) with `depends_on` edges,
per-task retries and timeouts, triggers (cron, file arrival, table update, continuous), and
notifications. Task values pass small results downstream as `{{tasks.<key>.values.<name>}}`.
**Repair runs** re-execute only failed and skipped tasks. A Jobs API payload:

```json
{
  "name": "daily-orders",
  "tasks": [
    {"task_key": "bronze", "notebook_task": {"notebook_path": "/Repos/prod/notebooks/bronze"}},
    {"task_key": "silver", "depends_on": [{"task_key": "bronze"}],
     "notebook_task": {"notebook_path": "/Repos/prod/notebooks/silver"}},
    {"task_key": "gold", "depends_on": [{"task_key": "silver"}],
     "sql_task": {"warehouse_id": "abc123", "query": {"query_id": "gold-refresh"}}}
  ],
  "schedule": {"quartz_cron_expression": "0 0 3 * * ?", "timezone_id": "UTC"},
  "max_concurrent_runs": 1
}
```

The scheduling semantics - dependency order, retries, and a repair run that skips what already
succeeded - fit in a few lines of standard-library Python:

In [ ]:
from graphlib import TopologicalSorter

DAG = {"bronze": set(), "dim_customers": set(), "silver": {"bronze"}, "gold": {"silver", "dim_customers"}}


def run_job(dag, task_fn, max_retries=1, previous=None):
    """Run tasks in dependency order; a repair run reuses previous successes."""
    state = dict(previous or {})
    for task in TopologicalSorter(dag).static_order():
        if state.get(task) == "SUCCESS":
            continue
        if any(state.get(upstream) != "SUCCESS" for upstream in dag[task]):
            state[task] = "SKIPPED"
            continue
        for attempt in range(1 + max_retries):
            try:
                task_fn(task, attempt)
                state[task] = "SUCCESS"
                break
            except RuntimeError:
                state[task] = "FAILED"
    return state


executed = []


def flaky(task, attempt):
    executed.append(task)
    if task == "silver":  # a bug that retries cannot fix
        raise RuntimeError("schema mismatch")


first_run = run_job(DAG, flaky)
print("run 1:", first_run)
assert first_run["silver"] == "FAILED" and first_run["gold"] == "SKIPPED"

executed.clear()
repair = run_job(DAG, lambda task, attempt: executed.append(task), previous=first_run)  # after the fix
print("repair run executed only:", executed)
assert executed == ["silver", "gold"] and set(repair.values()) == {"SUCCESS"}

Common mistakes: no `max_concurrent_runs`, so a slow 03:00 run overlaps the 04:00 one; passing
data through task values (they are for small scalars); rerunning a whole job instead of repairing
it; and a separate cluster per task instead of a shared job cluster.

## How do you manage Databricks resources as code?

**Databricks Asset Bundles** keep jobs, pipelines, and their files in Git: `databricks.yml`
declares targets (`dev`, `prod`), variables, and resources. **Terraform** owns long-lived
account-level objects (metastores, storage credentials, external locations, catalogs, cluster
policies, service principals).

```yaml
# databricks.yml
bundle: acme_lakehouse
workspace:
  host: https://acme.cloud.databricks.com
variables:
  catalog:
    default: acme_dev
resources:
  jobs:
    daily_orders:
      name: daily-orders-${bundle.target}
      schedule:
        quartz_cron_expression: "0 0 3 * * ?"
        timezone_id: UTC
      tasks:
        - task_key: bronze
          notebook_task:
            notebook_path: ./notebooks/bronze_orders.py
        - task_key: silver
          depends_on:
            - task_key: bronze
          notebook_task:
            notebook_path: ./notebooks/silver_orders.py
```

```bash
databricks bundle validate -t prod
databricks bundle deploy -t prod
databricks bundle run daily_orders -t prod
```

```hcl
# Terraform for workspace-level resources
resource "databricks_job" "daily_orders" {
  name = "daily-orders"
  task {
    task_key = "bronze"
    notebook_task { notebook_path = "/Repos/prod/notebooks/bronze" }
  }
}
```

Common mistakes: editing jobs in the UI and having the next `bundle deploy` revert them,
committing secrets into `databricks.yml`, and letting a bundle and Terraform manage the same job.
Keep Terraform state remote and locked.

## How do you query data with Databricks SQL?

A SQL warehouse is serverless or classic SQL compute sized from 2X-Small to 4X-Large (bigger =
one query faster) with a cluster count that scales out for concurrency, auto-stop, a result
cache, and Photon by default. Materialized views precompute repeated aggregates; query federation
reads external databases through a connection.

Common mistakes: one giant warehouse for dashboards, extracts, and ad hoc work (sized for the
heaviest, never auto-stopped), and assuming the result cache always applies - any write to a
source table or different query text is a miss.

## What is Photon and when does it help?

Photon is a vectorized C++ engine that replaces the JVM path for supported operators: Catalyst
still plans, Photon executes in columnar batches with SIMD and no GC pauses, and unsupported
operations fall back to Spark transparently.

+ **Helps**: large scans, filters, joins, aggregations, Delta `MERGE`, SQL warehouses.
+ **Does not help**: Python UDFs, RDD and Dataset APIs, stateful streaming, sub-second queries
  dominated by planning, and driver-side Python.
+ **Trade-off**: a higher DBU rate, often cheaper per job because it finishes faster.

Verify instead of assuming: the Spark UI draws Photon operators in a different colour, the query
profile shows the share of task time in Photon, and `product_features.is_photon` in
`system.billing.usage` tags Photon usage.

## How do you lay out and maintain Delta tables?

Delta keeps per-file min/max statistics (first 32 columns) for **data skipping**; target roughly
100 MB-1 GB per file.

+ **Liquid clustering** (`CLUSTER BY (cols)`, up to four keys) is the default for new tables:
  incremental `OPTIMIZE`, changeable keys, and `CLUSTER BY AUTO` to let predictive optimization
  pick keys from the workload.
+ **Partitioning** only for a low-cardinality column nearly every query filters; never by
  `customer_id`. It cannot be combined with liquid clustering.
+ **ZORDER** is the older co-location technique, incompatible with clustering.

Maintenance: `OPTIMIZE` compacts (and clusters), `VACUUM` deletes unreferenced files past
retention (ending time travel and change-feed history for that window - preview with `DRY RUN`),
`ANALYZE` collects optimizer statistics. **Predictive optimization** runs all three for managed
tables on serverless compute, so a hand-written hourly `OPTIMIZE` on top of it pays twice.

```sql
CREATE OR REPLACE TABLE acme.gold.fct_orders (
  order_id STRING NOT NULL, customer_id STRING, order_date DATE, amount DECIMAL(18, 2)
)
USING DELTA
CLUSTER BY (order_date, customer_id);

OPTIMIZE acme.gold.fct_orders;                      -- incremental
ALTER TABLE acme.gold.fct_orders CLUSTER BY (customer_id, order_date);
OPTIMIZE acme.gold.fct_orders FULL;                 -- recluster existing data after a key change
ALTER TABLE acme.gold.fct_orders CLUSTER BY AUTO;

ANALYZE TABLE acme.gold.fct_orders COMPUTE STATISTICS FOR ALL COLUMNS;
VACUUM acme.gold.fct_orders RETAIN 168 HOURS DRY RUN;
ALTER TABLE acme.gold.fct_orders SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '30 days');
ALTER TABLE acme.gold.fct_orders ENABLE PREDICTIVE OPTIMIZATION;
```

Compaction on a local Delta table - streaming-style small appends, then one `OPTIMIZE`:

In [ ]:
for minute in range(6):  # six micro-batches, six tiny files
    write_deltalake(
        orders_path,
        pa.table({"order_id": [f"s{minute}"], "status": ["placed"], "amount": [10.0], "source_seq": [1]}),
        mode="append",
    )

table = DeltaTable(orders_path)
files_before = len(table.file_uris())
rows_before = table.to_pyarrow_table().num_rows
result = table.optimize.compact()
table = DeltaTable(orders_path)

print(f"files: {files_before} -> {len(table.file_uris())}", {k: result[k] for k in ("numFilesAdded", "numFilesRemoved")})
assert len(table.file_uris()) == 1 and table.to_pyarrow_table().num_rows == rows_before

Common mistakes: changing `CLUSTER BY` without `OPTIMIZE ... FULL` (old data keeps the old
layout, so clustering "does not work"), vacuuming inside a time-travel window you rely on, and
never compacting a streaming table.

## How do you handle schema evolution and Change Data Feed?

With `delta.enableChangeDataFeed = true`, `table_changes('catalog.schema.table', start_version)`
returns changed rows tagged `insert`, `update_preimage`, `update_postimage`, or `delete`, with
`_commit_version` and `_commit_timestamp` - the natural source for incremental silver and gold
jobs. Only changes made while the property was on are recorded, and `VACUUM` bounds the history.

```sql
SELECT order_id, _change_type, _commit_version
FROM table_changes('acme.silver.orders', 1)
WHERE _change_type IN ('insert', 'update_postimage');
```

The `MERGE` above was committed with the feed enabled, so its changes can be read back:

In [ ]:
changes = pa.table(DeltaTable(orders_path).load_cdf(starting_version=1, ending_version=1).read_all())
rows = sorted(
    (row["order_id"], row["_change_type"])
    for row in changes.select(["order_id", "_change_type"]).to_pylist()
)
print(rows)
assert rows == [("o2", "update_postimage"), ("o2", "update_preimage"), ("o4", "insert")]

Schema evolution: `mergeSchema` lets a write add columns, and
`spark.databricks.delta.schema.autoMerge.enabled` extends that to `MERGE`. Column mapping
decouples physical from logical names (enabling renames); type widening records int-to-long as
metadata. Each upgrades the table protocol, so older readers can no longer read the table.

Rules that prevent incidents: add columns as nullable, never reuse a name for a new meaning, and
treat a rename as a migration (backfill plus compatibility view).

## How do you implement CI/CD for Databricks?

```text
1. Repos: notebooks and jobs live in Git (Databricks Repos or external CI checkout).
2. Validate: databricks bundle validate in each PR; run unit tests with pytest on pure PySpark transforms.
3. Deploy: databricks bundle deploy -t staging|prod from CI, promoting the same artifact.
4. Test on real data: run the job against a staging catalog, then compare row counts and business metrics.
5. Guardrails: Unity Catalog grants, job permissions, and cluster policies are code, reviewed like application code.
```

Keep transformations in importable modules tested with `pytest` on small DataFrames, notebooks
as thin wrappers, and job definitions in a bundle. Pull requests run `bundle validate` and unit
tests; merges deploy the same artifact to staging, then production. CI authenticates as a service
principal, never a personal token. Never point CI at the production catalog.

## What security and access-mode concepts matter?

+ **Access modes** - dedicated (formerly single-user), standard (formerly shared), and serverless; the mode determines which Spark features and languages are available and how users are isolated.
+ **Identity** - users, groups, service principals, and managed identities for jobs; avoid personal access tokens in production.
+ **Secrets** - Databricks secret scopes backed by Key Vault/Secrets Manager; never put credentials in notebooks or cluster env vars.
+ **Network** - private link/VNet injection, IP access lists, and no-public-IP clusters for regulated environments.
+ **Audit** - `system.access.audit` records who read, wrote, or changed what.

**Standard** mode lets many users share a cluster with isolation (Python, SQL, Scala);
**dedicated** mode assigns it to one user or group (adds R, ML runtimes, GPUs). Row filters and
column masks need standard mode on a supported runtime. Jobs should run as service principals,
secrets come from scopes (`{{secrets/scope/key}}`), and Unity Catalog governs data access - not
who can attach to a cluster, which is what cluster policies and audit review cover.

## How do you control and measure Databricks cost?

Cost has parts billed by different parties: DBUs to Databricks at a SKU rate (jobs, all-purpose,
SQL, pipelines, Photon, serverless), and VMs, disks, storage, and egress to the cloud provider on
classic compute. A classic run costs roughly `nodes x DBUs per node-hour x hours x price per DBU`
plus the VM bill for the same time.

The levers, roughly by size: move scheduled work to job compute, spot with fallback for batch,
right-sized autoscaling and auto-stop everywhere, Photon and liquid clustering so jobs finish
sooner, separate warehouses per workload, and tags by team and cost center.

Measure with system tables. `system.billing.usage` nests resource ids under `usage_metadata`
(`cluster_id`, `job_id`, `job_run_id`, `warehouse_id`, `dlt_pipeline_id`); jobs on all-purpose
compute have no `job_id`, so attribute them by cluster. `system.billing.list_prices` is a price
history - join on SKU and the price valid at the usage date. Here both tables are rebuilt as
small local views with the same shape, so the attribution query runs for real:

In [ ]:
from datetime import date

from pyspark.sql import Row

usage_rows = [
    (date(2026, 10, 1), "JOBS_COMPUTE", 27.0, Row(cluster_id="c-etl", job_id="j-orders", warehouse_id=None), {"team": "data-platform"}),
    (date(2026, 10, 1), "ALL_PURPOSE_COMPUTE", 40.0, Row(cluster_id="c-shared", job_id=None, warehouse_id=None), {"team": "analytics"}),
    (date(2026, 10, 2), "JOBS_COMPUTE", 25.0, Row(cluster_id="c-etl", job_id="j-orders", warehouse_id=None), {"team": "data-platform"}),
    (date(2026, 10, 2), "SQL_COMPUTE", 12.0, Row(cluster_id=None, job_id=None, warehouse_id="w-bi"), {"team": "analytics"}),
]
spark.createDataFrame(
    usage_rows,
    "usage_date date, sku_name string, usage_quantity double, "
    "usage_metadata struct<cluster_id:string,job_id:string,warehouse_id:string>, custom_tags map<string,string>",
).createOrReplaceTempView("billing_usage")

spark.createDataFrame(
    [  # illustrative prices; JOBS_COMPUTE changed on 2 October
        ("JOBS_COMPUTE", date(2026, 1, 1), date(2026, 10, 2), Row(effective_list=Row(default=0.40))),
        ("JOBS_COMPUTE", date(2026, 10, 2), None, Row(effective_list=Row(default=0.30))),
        ("ALL_PURPOSE_COMPUTE", date(2026, 1, 1), None, Row(effective_list=Row(default=1.00))),
        ("SQL_COMPUTE", date(2026, 1, 1), None, Row(effective_list=Row(default=0.70))),
    ],
    "sku_name string, price_start_time date, price_end_time date, pricing struct<effective_list:struct<default:double>>",
).createOrReplaceTempView("list_prices")

cost = spark.sql("""
SELECT coalesce(u.usage_metadata.job_id, u.usage_metadata.warehouse_id, u.usage_metadata.cluster_id) AS resource,
       u.custom_tags['team'] AS team,
       SUM(u.usage_quantity) AS dbus,
       ROUND(SUM(u.usage_quantity * p.pricing.effective_list.default), 2) AS list_cost
FROM billing_usage u
JOIN list_prices p
  ON p.sku_name = u.sku_name
 AND p.price_start_time <= u.usage_date
 AND (p.price_end_time IS NULL OR p.price_end_time > u.usage_date)
GROUP BY ALL
ORDER BY list_cost DESC
""")
cost.show()
by_resource = {row["resource"]: row["list_cost"] for row in cost.collect()}
assert by_resource == {"c-shared": 40.0, "j-orders": 18.3, "w-bi": 8.4}  # 27 x 0.40 + 25 x 0.30

`system.query.history` is the performance counterpart (compute under the `compute` struct,
plus `read_bytes`, `pruned_files`, `shuffle_read_bytes`, `spilled_local_bytes`,
`from_result_cache`, `query_tags`). Read its durations together to name the bucket before
changing anything - reads too much, does too much per row, or waits for compute:

In [ ]:
def diagnose(stmt):
    """Map one query-history row to the lever that addresses it."""
    if stmt["waiting_at_capacity_duration_ms"] > stmt["execution_duration_ms"]:
        return "queued: add warehouse clusters or isolate the workload"
    if stmt["spilled_local_bytes"] > 0.1 * stmt["read_bytes"]:
        return "spilling: reduce what operators hold, or give them more memory"
    if not stmt["from_result_cache"] and stmt["pruned_files"] < stmt["read_files"] * 0.5:
        return "scanning: fix layout (clustering keys) or the predicate"
    return "healthy: look at the query shape or Photon coverage"


history = [
    {"statement_id": "q1", "waiting_at_capacity_duration_ms": 90_000, "execution_duration_ms": 4_000,
     "spilled_local_bytes": 0, "read_bytes": 10**9, "read_files": 40, "pruned_files": 400, "from_result_cache": False},
    {"statement_id": "q2", "waiting_at_capacity_duration_ms": 0, "execution_duration_ms": 600_000,
     "spilled_local_bytes": 5 * 10**10, "read_bytes": 2 * 10**11, "read_files": 900, "pruned_files": 100, "from_result_cache": False},
    {"statement_id": "q3", "waiting_at_capacity_duration_ms": 0, "execution_duration_ms": 300_000,
     "spilled_local_bytes": 0, "read_bytes": 5 * 10**11, "read_files": 20_000, "pruned_files": 10, "from_result_cache": False},
]
verdicts = {stmt["statement_id"]: diagnose(stmt) for stmt in history}
for statement_id, verdict in verdicts.items():
    print(statement_id, "->", verdict)
assert [v.split(":")[0] for v in verdicts.values()] == ["queued", "spilling", "scanning"]

For a before-and-after, tag the sessions (`query_tags`), record DBUs per run, p50/p95 duration,
bytes read, shuffle, and spill, change one thing, and rerun on comparable data. If wall clock fell
but DBUs rose proportionally, you bought latency with money.

## How do you tune joins, shuffles and skew on Databricks?

AQE is on by default: it coalesces post-shuffle partitions (so leave
`spark.sql.shuffle.partitions` alone), switches sort-merge to broadcast joins when a side turns out
small, and splits skewed join partitions above both `skewedPartitionThresholdInBytes` and
`skewedPartitionFactor` times the median. Hints handle what AQE cannot: `BROADCAST` when statistics
are missing, `SKEW` to name the skewed relation and columns. Salting is the manual fallback - see
the [Spark Tuning](spark-tuning.ipynb) notebook for a runnable version.

```sql
SET spark.sql.adaptive.skewJoin.enabled = true;
SET spark.sql.adaptive.advisoryPartitionSizeInBytes = 128m;

SELECT /*+ BROADCAST(d) */ f.order_id, f.amount, d.segment
FROM acme.gold.fct_orders f
JOIN acme.gold.dim_customer d ON d.customer_id = f.customer_id;

SELECT /*+ SKEW('f', 'customer_id') */ f.customer_id, COUNT(*) AS orders
FROM acme.gold.fct_orders f
JOIN acme.gold.dim_customer d ON d.customer_id = f.customer_id
GROUP BY f.customer_id;
```

Read the profile first: one join task far above the rest is skew; a huge shuffle on both sides
may want a broadcast; heavy spill wants less data before the operator. Common mistakes:
broadcasting a table that is not small, salting before checking AQE, and treating a uniformly
slow stage (too few partitions) as skew.

## How do you size and isolate compute?

Type sets the rate, size sets speed and hourly cost, isolation decides who can affect whom. A
good production default: job cluster or serverless job, Photon, an autoscaling range from the
widest stage (tasks / cores per worker / target duration), spot with on-demand fallback for batch,
and a driver sized for what the job collects. Cluster policies constrain node types, max workers,
Photon, auto-termination, and tags so sizing is consistent:

```json
{
  "cluster_name": "etl-daily",
  "policy_id": "0011AABB22CC33DD",
  "spark_version": "15.4.x-scala2.12",
  "node_type_id": "m5d.4xlarge",
  "driver_node_type_id": "m5d.2xlarge",
  "autoscale": {"min_workers": 4, "max_workers": 32},
  "aws_attributes": {
    "availability": "SPOT_WITH_FALLBACK",
    "zone_id": "auto",
    "first_on_demand": 1
  },
  "runtime_engine": "PHOTON",
  "data_security_mode": "USER_ISOLATION",
  "autotermination_minutes": 20,
  "custom_tags": {"team": "data-platform", "cost-center": "analytics"},
  "spark_conf": {"spark.sql.adaptive.enabled": "true"}
}
```

Compare total cost, not unit rates: serverless looks pricier per DBU because it includes the
infrastructure, and an all-purpose cluster also idles and bills.

## How does Databricks compare with Snowflake and BigQuery?

| Aspect | Databricks | Snowflake | BigQuery |
|---|---|---|---|
| Engine | Spark + Photon, plus SQL warehouses | proprietary SQL engine | serverless SQL engine |
| Table format | Delta (and Iceberg) on your storage | Snowflake-managed micro-partitions (Iceberg tables available) | BigQuery-managed storage (BigLake/Iceberg available) |
| Strength | large-scale Spark/ML, notebooks, streaming, open formats | SQL analytics, ease of use, governed sharing | serverless simplicity, tight GCP integration |
| Cost model | DBUs + cloud VM/storage | credits + storage | on-demand/edition slots + storage |
| Governance | Unity Catalog | native RBAC + policies | IAM + Dataplex/policy tags |

Separate the engine from the billing model: Spark plus Photon over open files in your storage
billed in DBUs plus cloud VMs; a proprietary engine over managed micro-partitions billed in
credits; a serverless engine over Google-managed storage billed on demand or by slots. All three
read open formats now, so the differentiator is where data lives, who governs it, and what else
must run against it. Choose Databricks for Spark-scale engineering, streaming, ML, and open
formats; Snowflake or BigQuery when the workload is mostly SQL analytics and managed simplicity
wins. Many companies run two - say so rather than declaring a winner. The deciding metric is cost
per pipeline run at the required latency, not cost per terabyte scanned.

## How do you migrate from on-prem Spark and Hive?

+ **Storage first** - copy HDFS/Hive data to object storage (DistCp, cloud transfer service), then register it as external Delta/Iceberg tables or migrate in place.
+ **Metastore** - move Hive metastore objects into Unity Catalog; use the migration tooling to upgrade tables to Delta and fix unsupported types.
+ **Code** - replace `hive`-specific SQL with Spark SQL equivalents, remove `HiveContext`, and convert Python UDFs to built-in functions.
+ **Jobs** - port Oozie/Airflow DAGs to Databricks Workflows or keep Airflow and call Databricks jobs via the operator.
+ **Validate** - compare row counts and checksums between old and new tables, then run both paths in parallel for a period before cutover.

Order matters: bytes first, then the catalog, then code, schedules, and validation - a
metastore migrated before storage points at paths that no longer exist. Expect Hive SQL
differences (implicit casts, non-deterministic ordering, `INSERT OVERWRITE` semantics), rewrite
Python UDFs as built-ins for Photon, migrate grants along with tables, and run old and new in
parallel before cutover.

## What is MLflow and how is it used on Databricks?

MLflow tracks experiments (parameters, metrics, artifacts), packages models with signatures, and
versions them in a registry - on Databricks the registry is Unity Catalog
(`catalog.schema.model_name`):

```python
import mlflow

mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run(run_name="churn-baseline"):
    mlflow.log_params({"max_depth": 8, "n_estimators": 200})
    mlflow.log_metric("auc", 0.912)
    mlflow.sklearn.log_model(model, artifact_path="model",
                             registered_model_name="acme.ml.churn_classifier")
```

For data engineers the contract is the features: the same definitions must feed training and
low-latency serving, and training rows must use **point-in-time** feature values known at the
label's timestamp, or the model learns from the future. Log table references, not whole datasets,
and register models with a signature.

## What are common Databricks interview traps?

+ Saying "Databricks is just hosted Spark" - it also includes Delta, Unity Catalog, workflows, SQL warehouses, and governance.
+ Running production pipelines on an all-purpose cluster.
+ Using `collect()` or `toPandas()` on large data and blaming the cluster.
+ Calling Python UDFs on huge datasets and expecting Photon to fix it.
+ Over-partitioning a Delta table, then wondering why the file listing is slow.
+ Confusing `VACUUM` retention with time travel: vacuuming inside the retention window breaks time travel.
+ Forgetting that Unity Catalog changes the namespace to three levels, so old two-level paths stop working.
+ Assuming autoscaling fixes skew - it cannot split a single hot partition.

Answer as two-part statements - the mechanism, then where it breaks: "autoscaling adds workers
for pending tasks, so it helps a widening job but cannot split a single hot partition." More to
have ready: `OPTIMIZE` is not free (it rewrites files and burns DBUs), serverless still needs
network paths to your storage, and a performance claim needs DBUs per run and the query profile,
not just wall clock.

## Databricks exercises

Standard-library models of the sizing, cost, and layout arithmetic above. The numbers are
illustrative, not a price list; the asserts check the formula.

### Try it 1. Pick a node family from the workload shape

**Task.** For each workload, choose a node family. Prefer storage-optimized when the job spills more than 20 percent of its input to local disk, memory-optimized when it shuffles at least twice its input size, and compute-optimized otherwise. The metric is `spill_gb / data_gb`.

**Hint.** Check the spill ratio first, because disk pressure dominates; then the shuffle ratio.

**Expected result.** `nightly_etl` is memory-optimized, `feature_math` is compute-optimized, and `log_parse` is storage-optimized.

In [ ]:
def pick_node_family(workload):
    if workload["spill_gb"] / workload["data_gb"] > 0.2:
        return "storage-optimized (spills to local disk)"
    if workload["shuffle_ratio"] >= 2:
        return "memory-optimized (large shuffles and caches)"
    return "compute-optimized (CPU-bound, little shuffle)"


workloads = {
    "nightly_etl": {"data_gb": 800, "shuffle_ratio": 4.0, "spill_gb": 120},
    "feature_math": {"data_gb": 200, "shuffle_ratio": 0.5, "spill_gb": 0},
    "log_parse": {"data_gb": 2000, "shuffle_ratio": 1.0, "spill_gb": 600},
}

picks = {}
for name, workload in workloads.items():
    picks[name] = pick_node_family(workload)
    print(f"{name:<14} -> {picks[name]}")

assert picks["nightly_etl"].startswith("memory-optimized")
assert picks["feature_math"].startswith("compute-optimized")
assert picks["log_parse"].startswith("storage-optimized")

### Try it 2. Compute DBU cost and compare job with all-purpose

**Task.** A job uses 8 workers plus a driver, at 4 DBUs per node-hour, and runs for 45 minutes. Compute the DBUs and the cost at an illustrative job rate of 0.40 per DBU and an all-purpose rate of 1.00 per DBU, then report the percentage saved by running on jobs compute.

**Hint.** `dbu_per_hour = (workers + 1) x DBUs per node-hour`, and `dbus = dbu_per_hour x runtime_hours`.

**Expected result.** 36 DBUs per hour, 27 DBUs for the run, 10.80 on jobs compute versus 27.00 on all-purpose, a saving of 60 percent.

In [ ]:
DBUS_PER_NODE_HOUR = 4.0
WORKERS = 8
RUNTIME_MINUTES = 45
JOB_PRICE_PER_DBU = 0.40          # illustrative example rates
ALL_PURPOSE_PRICE_PER_DBU = 1.00

nodes = WORKERS + 1
dbu_per_hour = nodes * DBUS_PER_NODE_HOUR
runtime_hours = RUNTIME_MINUTES / 60
dbus = dbu_per_hour * runtime_hours

job_cost = dbus * JOB_PRICE_PER_DBU
all_purpose_cost = dbus * ALL_PURPOSE_PRICE_PER_DBU
savings_pct = (all_purpose_cost - job_cost) / all_purpose_cost

print(f"{nodes} nodes x {DBUS_PER_NODE_HOUR} DBU/h = {dbu_per_hour:.0f} DBU/h")
print(f"{RUNTIME_MINUTES} min -> {dbus:.1f} DBUs")
print(f"jobs compute: {job_cost:.2f}")
print(f"all-purpose:  {all_purpose_cost:.2f}  (saving {savings_pct * 100:.0f}%)")

assert dbu_per_hour == 36.0
assert round(dbus, 2) == 27.0
assert round(job_cost, 2) == 10.80
assert round(all_purpose_cost, 2) == 27.00
assert round(savings_pct * 100, 1) == 60.0

### Try it 3. Decide partitioning versus liquid clustering

**Task.** Given a table size and the filter rate and cardinality of each column, return a layout plan. Partition only when the table is at least 100 GB and some column is filtered by at least 90 percent of queries with cardinality at most 10,000. Leave tables under 50 GB alone. Otherwise use liquid clustering on the two most-filtered columns.

**Hint.** Sort the columns by `filter_rate` to pick the clustering keys, and check the low-cardinality always-filtered condition first.

**Expected result.** The 900 GB events table partitions by `event_date`, the 900 GB clicks table clusters on `(order_ts, customer_id)`, and the 5 GB table gets no layout change.

In [ ]:
def choose_layout(size_gb, columns):
    low_cardinality_always = [
        name for name, stats in columns.items()
        if stats["filter_rate"] >= 0.9 and stats["cardinality"] <= 10_000
    ]
    if size_gb >= 100 and low_cardinality_always:
        return "partition by " + low_cardinality_always[0]
    if size_gb < 50:
        return "plain Delta, no layout change"
    top_two = sorted(columns, key=lambda name: columns[name]["filter_rate"], reverse=True)[:2]
    return "liquid clustering on (" + ", ".join(top_two) + ")"


events = {
    "event_date": {"cardinality": 730, "filter_rate": 1.0},
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.7},
    "country": {"cardinality": 40, "filter_rate": 0.1},
}
clicks = {
    "order_ts": {"cardinality": 8_000_000, "filter_rate": 0.9},
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.8},
    "country": {"cardinality": 40, "filter_rate": 0.05},
}
tiny = {
    "event_date": {"cardinality": 730, "filter_rate": 1.0},
    "customer_id": {"cardinality": 5_000_000, "filter_rate": 0.7},
}

plan_events = choose_layout(900, events)
plan_clicks = choose_layout(900, clicks)
plan_tiny = choose_layout(5, tiny)

print("events (900 GB):", plan_events)
print("clicks (900 GB):", plan_clicks)
print("tiny (5 GB):    ", plan_tiny)

assert plan_events == "partition by event_date"
assert plan_clicks == "liquid clustering on (order_ts, customer_id)"
assert plan_tiny == "plain Delta, no layout change"

### Try it 4. Detect skew and size a salt

**Task.** Given the row counts of the partitions in a shuffle stage, decide whether the stage is skewed. Treat a partition as skewed when its row count is at least three times the median. Then work out how many salt buckets the hot key needs so each bucket is no larger than a target number of rows.

**Hint.** Use `statistics.median` for the threshold and `math.ceil(max_rows / target_rows)` for the bucket count, capped at 256.

**Expected result.** The skewed example has a median of 100.5 and is detected; the balanced example is not; the hot key needs 3 buckets at a target of 5,000 rows.

In [ ]:
import math
import statistics


def detect_skew(partition_rows, factor=3.0):
    median = statistics.median(partition_rows)
    return max(partition_rows) / median >= factor, median


def salt_buckets(max_rows, target_rows, cap=256):
    return max(1, min(cap, math.ceil(max_rows / target_rows)))


skewed = [100, 98, 105, 101, 99, 100, 12_000, 103]
balanced = [100, 98, 105, 101, 99, 100, 103, 104]

skewed_flag, skewed_median = detect_skew(skewed)
balanced_flag, _ = detect_skew(balanced)
buckets = salt_buckets(max(skewed), target_rows=5000)

print(f"skewed median = {skewed_median:.1f}, max/median = {max(skewed) / skewed_median:.1f}")
print("skewed detected:", skewed_flag)
print("balanced is skewed:", balanced_flag)
print("salt buckets for the hot key:", buckets)

assert skewed_median == 100.5
assert skewed_flag is True
assert balanced_flag is False
assert buckets == 3

### Try it 5. Choose a target file count from table size

**Task.** A table is 2 TB and the target file size is 256 MB. Compute how many files it should have. Then, for a table currently split into 200,000 files averaging 10 MB each, decide whether it needs compaction, treating an average below half the target size as too small.

**Hint.** One TB is 1024 x 1024 MB. Compact when the current average file size is below `target_mb / 2`.

**Expected result.** 8,192 files at 256 MB each, a current average of 10 MB, and compaction is needed.

In [ ]:
import math

TARGET_FILE_MB = 256
TABLE_TB = 2.0
MB_PER_TB = 1024 * 1024

target_files = math.ceil(TABLE_TB * MB_PER_TB / TARGET_FILE_MB)

current_files = 200_000
current_avg_mb = (current_files * 10) / current_files
needs_compaction = current_avg_mb < TARGET_FILE_MB / 2

print(f"2 TB at {TARGET_FILE_MB} MB per file -> {target_files} files")
print(f"current average file size = {current_avg_mb:.0f} MB")
print("needs compaction:", needs_compaction)

assert target_files == 8192
assert current_avg_mb == 10.0
assert needs_compaction is True

### Try it 6. Compare a Photon run with a non-Photon run

**Task.** A job takes 60 minutes on a normal cluster at 32 DBUs per hour. With Photon it takes 24 minutes but consumes 30 percent more DBUs per hour. Compute both costs at 0.15 per DBU, the speedup, and whether Photon is cheaper per run.

**Hint.** Cost is `dbu_per_hour x minutes / 60 x price`. Photon can cost less per job even though it costs more per hour.

**Expected result.** 4.80 without Photon, 2.50 with Photon, a 2.50x speedup, and Photon is cheaper for this job.

In [ ]:
NON_PHOTON_DBU_PER_HOUR = 32.0
PHOTON_DBU_PER_HOUR = 41.6        # 30 percent more DBUs per hour
NON_PHOTON_MINUTES = 60
PHOTON_MINUTES = 24
PRICE_PER_DBU = 0.15


def job_cost(dbu_per_hour, minutes, price_per_dbu):
    return dbu_per_hour * (minutes / 60) * price_per_dbu


before = job_cost(NON_PHOTON_DBU_PER_HOUR, NON_PHOTON_MINUTES, PRICE_PER_DBU)
after = job_cost(PHOTON_DBU_PER_HOUR, PHOTON_MINUTES, PRICE_PER_DBU)
speedup = NON_PHOTON_MINUTES / PHOTON_MINUTES

print(f"without Photon: {before:.2f} for {NON_PHOTON_MINUTES} min")
print(f"with Photon:    {after:.2f} for {PHOTON_MINUTES} min")
print(f"speedup = {speedup:.2f}x, Photon is cheaper per run = {after < before}")

assert round(before, 2) == 4.80
assert round(after, 2) == 2.50
assert speedup == 2.5
assert after < before

### Try it 7. Estimate an autoscaling range from concurrency

**Task.** A stage has 2,000 shuffle tasks at 2 minutes each and must finish in 20 minutes. With 8 cores per worker, compute the concurrent slots needed, the steady worker count, and an autoscaling range: minimum at 20 percent of steady (floor of 2) and maximum at 20 percent headroom over steady.

**Hint.** `slots = ceil(total_task_minutes / target_minutes)`, then `workers = ceil(slots / cores_per_worker)`.

**Expected result.** 4,000 task-minutes, 200 slots, 25 steady workers, and a range of 5 to 30 workers.

In [ ]:
import math

TASKS = 2000
MINUTES_PER_TASK = 2.0
TARGET_STAGE_MINUTES = 20
CORES_PER_WORKER = 8
HEADROOM = 1.2
MIN_FLOOR = 2

total_task_minutes = TASKS * MINUTES_PER_TASK
concurrent_slots = math.ceil(total_task_minutes / TARGET_STAGE_MINUTES)
steady_workers = math.ceil(concurrent_slots / CORES_PER_WORKER)
max_workers = math.ceil(steady_workers * HEADROOM)
min_workers = max(MIN_FLOOR, math.ceil(steady_workers * 0.2))

print(f"total task minutes = {total_task_minutes:.0f}")
print(f"concurrent slots needed = {concurrent_slots}")
print(f"steady workers = {steady_workers}")
print(f"autoscaling range = {min_workers} to {max_workers}")

assert concurrent_slots == 200
assert steady_workers == 25
assert min_workers == 5
assert max_workers == 30

## Real world use cases for Databricks

Read each case as an answer template: what arrives, which layer handles it, which feature does
the hard part, and the number that proved it worked.

1. **Retail lakehouse.** S3 JSON -> Auto Loader (bronze) -> dedupe and type (silver) -> facts and
   dimensions (gold) -> SQL warehouse dashboards. One pipeline feeds BI and data science; Unity
   Catalog masks and row filters govern PII.
2. **Streaming CDC.** Debezium events from Kafka are merged into silver Delta tables per
   micro-batch with `foreachBatch` + `MERGE`: exactly-once upserts and deletes on the same
   platform as batch.
3. **Hive migration.** Hive tables move to object storage and become Delta, Unity Catalog replaces
   the metastore's grants, Oozie jobs become Workflows.
4. **ML platform.** Nightly Delta feature tables registered in Unity Catalog feed MLflow-tracked
   training; serving reads the same features from an online store, with lineage from table to
   model.
5. **Cost program.** Scheduled work moves to job clusters with spot fallback, hot tables get
   Photon and liquid clustering, warehouses are right-sized, and `system.billing.usage` is
   reviewed monthly by cost-center tag.

If you can explain the plane and billing split, choose a layout from the query pattern, prove a
claim from the system tables, and design an idempotent, repairable task DAG, you can answer most
Databricks interviews.